In [1]:
import os
import pandas as pd
from prettytable import PrettyTable

from lib.uncertinay_rat import SimulateRat

/Users/felix/MSE/03_projects/MT/zz_code/04_experiments/03_analysis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DS_FEVER = '../02_data/2026-03-12/fever'
DS_HOTPOTQA = '../02_data/2026-03-12/hotpotqa'
DS_NQ = '../02_data/2026-03-12/nq'

In [3]:
def get_retrieval_success(row):
    ret_set = set([f"{r['document_id']}:{r['index']}" for r in row['retrieved']])
    ref_set = set([f"{r['document_id']}:{r['index']}" for r in row['reference']])

    return set(ref_set) <= set(ret_set)

def set_abstain(row):
    return (row['generated_answer'] == 'I DO NOT KNOW') or (row['generated_answer'] == 'NOT ENOUGH INFO')

def set_task_success(row):
    return row['correct_answer']

def set_generator_success(row):
    if row['retriever_success'] == True:
        return row['task_success']
    else:
        return row['abstain']

In [4]:
results_all = {}
success_rates = {}
samples_all = {}

simulate = SimulateRat()

for g in os.listdir(f"{DS_FEVER}"):
    for experiment in os.listdir(f"{DS_FEVER}/{g}/"):
        id = experiment.split('_')
        results = pd.read_json(f'{DS_FEVER}/{g}/{experiment}/results.json')

        results['dataset'] = 'fever'
        results['generator'] = g
        results['retriever_strategy'] = experiment

        results['correct_query'] = True
        results['retriever_success'] = results.apply(get_retrieval_success, axis=1)
        results['abstain'] = results.apply(set_abstain, axis=1)
        results['task_success'] = results.apply(set_task_success, axis=1)
        results['generator_success'] = results.apply(set_generator_success, axis=1)

        results_all[f"fever_{g}_{experiment}"] = results
        success_rates[f"fever_{g}_{experiment}"], samples_all[f"fever_{g}_{experiment}"] = simulate.compute_uncertainty(results)


for g in os.listdir(f"{DS_HOTPOTQA}"):
    for experiment in os.listdir(f"{DS_HOTPOTQA}/{g}/"):
        id = experiment.split('_')
        results = pd.read_json(f'{DS_HOTPOTQA}/{g}/{experiment}/results.json')

        results['dataset'] = 'hotpotqa'
        results['generator'] = g
        results['retriever_strategy'] = experiment

        results['correct_query'] = True
        results['retriever_success'] = results.apply(get_retrieval_success, axis=1)
        results['abstain'] = results.apply(set_abstain, axis=1)
        results['task_success'] = results.apply(set_task_success, axis=1)
        results['generator_success'] = results.apply(set_generator_success, axis=1)
                            
        results_all[f"hotpotqa_{g}_{experiment}"] = results
        success_rates[f"hotpotqa_{g}_{experiment}"], samples_all[f"hotpotqa_{g}_{experiment}"] = simulate.compute_uncertainty(results)

for g in os.listdir(f"{DS_NQ}"):
    for experiment in os.listdir(f"{DS_NQ}/{g}/"):
        try:
            id = experiment.split('_')
            results = pd.read_json(f'{DS_NQ}/{g}/{experiment}/results.json')

            results['dataset'] = 'nq'
            results['generator'] = g
            results['retriever_strategy'] = experiment

            results['correct_query'] = True
            results['retriever_success'] = results.apply(get_retrieval_success, axis=1)
            results['abstain'] = results.apply(set_abstain, axis=1)
            results['task_success'] = results.apply(set_task_success, axis=1)
            results['generator_success'] = results.apply(set_generator_success, axis=1)
                                
            results_all[f"nq_{g}_{experiment}"] = results
            success_rates[f"nq_{g}_{experiment}"], samples_all[f"nq_{g}_{experiment}"] = simulate.compute_uncertainty(results)
        except:
            continue

out = pd.concat((df for df in results_all.values()), ignore_index=True)


In [5]:
t = PrettyTable(field_names=['Dataset', 'Generator', 'Retriever Strategy', 'P(R=1)', 'P(A=1)', 'P(T=1)', 'P(G=1)'])

for experiment, success_rate in success_rates.items():
    id = experiment.split('_')

    df = out.loc[(out['dataset'] == id[0]) & (out['generator'] == id[1]) & (out['retriever_strategy'] == id[2])]

    t.add_row([
        id[0],
        id[1],
        id[2],
        f"{df['retriever_success'].mean():.2f}",
        f"{df['abstain'].mean():.2f}",
        f"{df['task_success'].mean():.2f}",
        f"{df['generator_success'].mean():.2f}"
    ]) 
 
t

Dataset,Generator,Retriever Strategy,P(R=1),P(A=1),P(T=1),P(G=1)
fever,gemma3,dense,0.60,0.18,0.78,0.74
fever,gemma3,sparse,0.46,0.22,0.74,0.64
fever,gemma3,hybrid,0.66,0.10,0.86,0.70
fever,apertus,dense,0.60,0.13,0.75,0.61
fever,apertus,sparse,0.46,0.12,0.76,0.50
fever,apertus,hybrid,0.66,0.09,0.80,0.61
fever,qwen,empty,0.00,0.68,0.28,0.68
fever,qwen,oracle,1.00,0.04,0.92,0.92
fever,qwen,dense,0.60,0.20,0.75,0.74
fever,qwen,sparse,0.46,0.25,0.71,0.66


In [6]:
t = PrettyTable(field_names=['Dataset', 'Generator', 'Retriever Strategy', 'P(R=1)', 'P(A=1)', 'P(T=1)', 'P(G=1)'])

for experiment, success_rate in success_rates.items():
    id = experiment.split('_')

    df = out.loc[(out['dataset'] == id[0]) & (out['generator'] == id[1]) & (out['retriever_strategy'] == id[2])]

    t.add_row([
        id[0],
        id[1],
        id[2],
        f"{success_rate['r']['mean']:.2%} ± {success_rate['r']['std']:.4%}",
        f"{success_rate['a']['mean']:.2%} ± {success_rate['a']['std']:.4%}",
        f"{success_rate['t']['mean']:.2%} ± {success_rate['t']['std']:.4%}",
        f"{success_rate['g']['mean']:.2%} ± {success_rate['g']['std']:.4%}",
    ]) 
 
t

Dataset,Generator,Retriever Strategy,P(R=1),P(A=1),P(T=1),P(G=1)
fever,gemma3,dense,60.34% ± 0.4890%,18.02% ± 0.3852%,78.42% ± 0.4123%,73.59% ± 0.4381%
fever,gemma3,sparse,45.55% ± 0.4985%,22.07% ± 0.4160%,74.37% ± 0.4376%,64.35% ± 0.4768%
fever,gemma3,hybrid,65.55% ± 0.4747%,9.62% ± 0.2951%,86.34% ± 0.3439%,70.12% ± 0.4540%
fever,apertus,dense,60.34% ± 0.4890%,13.50% ± 0.3416%,75.34% ± 0.4312%,60.81% ± 0.4846%
fever,apertus,sparse,45.55% ± 0.4985%,12.45% ± 0.3303%,75.93% ± 0.4274%,50.40% ± 0.4966%
fever,apertus,hybrid,65.55% ± 0.4747%,8.91% ± 0.2846%,80.10% ± 0.3997%,61.40% ± 0.4825%
fever,qwen,empty,0.01% ± 0.0099%,68.15% ± 0.4666%,28.34% ± 0.4480%,68.15% ± 0.4667%
fever,qwen,oracle,99.99% ± 0.0099%,4.47% ± 0.2083%,91.59% ± 0.2785%,91.58% ± 0.2786%
fever,qwen,dense,60.34% ± 0.4890%,19.94% ± 0.4003%,75.13% ± 0.4328%,74.36% ± 0.4341%
fever,qwen,sparse,45.55% ± 0.4985%,24.61% ± 0.4318%,70.82% ± 0.4551%,65.65% ± 0.4729%


In [7]:
t = PrettyTable(field_names=['Dataset', 'Generator', 'Retriever Strategy', 'P(A=1|R0)', 'P(A=1|R1)', 'P(T=1|R0,A0)', 'P(T=1|R1,A0)'])

for experiment, success_rate in success_rates.items():
    id = experiment.split('_')

    df = out.loc[(out['dataset'] == id[0]) & (out['generator'] == id[1]) & (out['retriever_strategy'] == id[2])]

    t.add_row([
        id[0],
        id[1],
        id[2],  
        # f"{success_rate['a_r0']['mean']:.2f} ± {success_rate['a_r0']['std']:.4f}",
        f"{success_rate['a_r0']['mean']:.2f}",
        # f"{success_rate['a_r1']['mean']:.2f} ± {success_rate['a_r1']['std']:.4f}",
        f"{success_rate['a_r1']['mean']:.2f}",
        # f"{success_rate['t_r0_a0']['mean']:.2f} ± {success_rate['t_r0_a0']['std']:.4f}",
        f"{success_rate['t_r0_a0']['mean']:.2f}",
        # f"{success_rate['t_r1_a0']['mean']:.2f} ± {success_rate['t_r1_a0']['std']:.4f}",
        f"{success_rate['t_r1_a0']['mean']:.2f}",
    ]) 
 
t.get_latex_string()

'\\begin{tabular}{ccccccc}\r\nDataset & Generator & Retriever Strategy & P(A=1|R0) & P(A=1|R1) & P(T=1|R0,A0) & P(T=1|R1,A0) \\\\\r\nfever & gemma3 & dense & 0.42 & 0.02 & 0.94 & 0.96 \\\\\r\nfever & gemma3 & sparse & 0.39 & 0.02 & 0.94 & 0.96 \\\\\r\nfever & gemma3 & hybrid & 0.24 & 0.02 & 0.94 & 0.96 \\\\\r\nfever & apertus & dense & 0.25 & 0.06 & 0.81 & 0.90 \\\\\r\nfever & apertus & sparse & 0.20 & 0.04 & 0.83 & 0.91 \\\\\r\nfever & apertus & hybrid & 0.16 & 0.05 & 0.83 & 0.90 \\\\\r\nfever & qwen & empty & 0.68 & 0.50 & 0.89 & 0.50 \\\\\r\nfever & qwen & oracle & 0.50 & 0.04 & 0.50 & 0.96 \\\\\r\nfever & qwen & dense & 0.46 & 0.03 & 0.89 & 0.95 \\\\\r\nfever & qwen & sparse & 0.43 & 0.03 & 0.92 & 0.96 \\\\\r\nfever & qwen & hybrid & 0.27 & 0.03 & 0.91 & 0.95 \\\\\r\nhotpotqa & gemma3 & dense & 0.36 & 0.03 & 0.31 & 0.57 \\\\\r\nhotpotqa & gemma3 & sparse & 0.29 & 0.01 & 0.33 & 0.63 \\\\\r\nhotpotqa & gemma3 & hybrid & 0.27 & 0.02 & 0.33 & 0.61 \\\\\r\nhotpotqa & apertus & dense & 0